# Person Detection — YOLOv8 Fine-tuning

This notebook fine-tunes a **YOLOv8n** model on a person-detection dataset using custom data augmentation to improve robustness in challenging conditions (low light, fog, noise, occlusion).

---

## 1. Install Dependencies

Install the required libraries: `ultralytics` for YOLOv8 and `albumentations` for custom augmentation pipelines.

In [ ]:
!pip install ultralytics albumentations

## 2. Download Dataset

Download the person detection dataset from Roboflow in YOLOv8 format. The dataset contains annotated images of people and will be used for fine-tuning the pre-trained model.

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="xut9urYUaByBEdBxCDlQ")
project = rf.workspace("neens").project("coco-person-kgihg")
version = project.version(1)
dataset = version.download("yolov8")

## 3. Define Augmentations and Fine-tune

A custom augmentation pipeline is applied at training time to simulate real-world degraded conditions:

- **RandomBrightnessContrast** — simulates low-light or overexposed environments
- **RandomFog** — adds synthetic fog to improve robustness in poor visibility
- **GaussianBlur** — mimics motion blur or out-of-focus cameras
- **CoarseDropout** — occludes random patches to handle partial visibility
- **RandomShadow** — simulates shadows cast by objects or architecture
- **ToGray** — occasional grayscale conversion for sensor diversity
- **GaussNoise** — adds sensor noise common in low-quality cameras

The augmentation is injected into the training loop via the `on_train_batch_start` callback. The base model is **YOLOv8n** (nano), pre-trained on COCO, and fine-tuned for 30 epochs on the person dataset.

In [ ]:
import albumentations as A
from ultralytics import YOLO
import numpy as np
import torch

augmentation_pipeline = A.Compose([

    A.RandomBrightnessContrast(
        brightness_limit=(-0.5, -0.1),
        contrast_limit=(-0.3, 0.3),
        p=0.7
    ),

    # Updated API: fog_coef_range instead of fog_coef_lower/upper
    A.RandomFog(
        fog_coef_range=(0.2, 0.6),
        alpha_coef=0.1,
        p=0.5
    ),

    A.GaussianBlur(
        blur_limit=(3, 7),
        p=0.4
    ),

    # Updated API: num_holes_range, hole_height_range, hole_width_range
    A.CoarseDropout(
        num_holes_range=(2, 8),
        hole_height_range=(20, 40),
        hole_width_range=(20, 40),
        fill_value=0,
        p=0.4
    ),

    # Updated API: num_shadows_range instead of num_shadows_lower/upper
    A.RandomShadow(
        shadow_roi=(0, 0.5, 1, 1),
        num_shadows_range=(1, 3),
        shadow_dimension=5,
        p=0.3
    ),

    A.ToGray(p=0.2),

    # Updated API: std_range instead of var_limit
    A.GaussNoise(
        std_range=(0.1, 0.2),
        p=0.4
    ),

], bbox_params=A.BboxParams(
    format='yolo',
    label_fields=['class_labels'],
    min_visibility=0.3
))


def apply_augmentations(trainer):
    if not hasattr(trainer, 'batch') or trainer.batch is None:
        return

    images = trainer.batch['img']
    batch_size = images.shape[0]

    for i in range(batch_size):
        img = images[i].cpu().numpy().transpose(1, 2, 0)
        img = (img * 255).astype(np.uint8)

        batch_idx = trainer.batch['batch_idx']
        mask = (batch_idx == i)
        bboxes = trainer.batch['bboxes'][mask].cpu().numpy().tolist()
        class_labels = trainer.batch['cls'][mask].cpu().numpy().tolist()

        if len(bboxes) == 0:
            continue

        try:
            result = augmentation_pipeline(
                image=img,
                bboxes=bboxes,
                class_labels=class_labels
            )
            augmented = result['image'].astype(np.float32) / 255.0
            images[i] = torch.from_numpy(
                augmented.transpose(2, 0, 1)
            ).to(images.device)
        except Exception:
            pass

    trainer.batch['img'] = images


model = YOLO("yolov8n.pt")
model.add_callback("on_train_batch_start", apply_augmentations)

model.train(
    data=dataset.location + "/data.yaml",
    epochs=30,
    imgsz=640,
    lr0=0.001,
    batch=16,
    hsv_v=0.5,
    hsv_s=0.3,
    degrees=10.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    # 'blur' removed — not a valid model.train() parameter
)

## 4. Export to ONNX

Export the best checkpoint produced during training to **ONNX format** for deployment. ONNX makes the model portable across inference runtimes (OpenCV, ONNX Runtime, TensorRT, etc.) without depending on PyTorch.

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/runs/detect/train/weights/best.pt")
model.export(format="onnx", imgsz=640, opset=12)